# Clase 2 · Pre-clase — Bases de Datos Transaccionales y Analíticas + Componentes de una Bodega de Datos

**Qué haces antes de venir a clase (60-90 min):**
1. Lee este notebook completo.
2. Ejecuta las celdas de código para observar la estructura del dataset Saber 11.
3. Responde las **celdas de reflexión** (marcadas 🟡). Sube el notebook completado a Moodle antes de tu clase.

**Al terminar deberías poder:**
- Diferenciar una base de datos transaccional (OLTP) de una analítica (OLAP).
- Nombrar los componentes de una bodega de datos.
- Bosquejar un modelo estrella para Saber 11.
- Anticipar qué dimensiones tendrá tu proyecto grupal.

## 1. Bases de Datos Transaccionales vs. Analíticas

| Característica | OLTP (Transaccional) | OLAP (Analítica) |
|---|---|---|
| **Propósito** | Registrar operaciones del negocio en tiempo real | Analizar grandes volúmenes de datos históricos |
| **Carga de trabajo típica** | Muchas escrituras y lecturas pequeñas simultáneas | Consultas complejas de lectura sobre millones de filas |
| **Volumen de datos** | Gigabytes a terabytes; filas recientes | Terabytes a petabytes; acumulación histórica |
| **Latencia** | Milisegundos (tiempo real) | Segundos a minutos (análisis en batch o columnar) |
| **Esquema** | Normalizado (3FN o BCNF) para minimizar redundancia | Denormalizado (estrella o copo de nieve) para acelerar consultas |
| **Ejemplos** | Sistema bancario, e-commerce, ERP, inscripción universitaria | Data warehouse, BI corporativo, dashboards MEN, análisis Saber 11 |

En el mundo real ambas tecnologías **conviven y se complementan**: los sistemas OLTP capturan los eventos del negocio y los cargan periódicamente (vía ETL/ELT) hacia un almacén analítico OLAP. Sin el OLTP no hay datos frescos; sin el OLAP no hay análisis histórico a escala.

## 2. Modelos Dimensionales: Estrella y Copo de Nieve

Un **modelo dimensional** organiza los datos en torno a un hecho central (medidas numéricas) rodeado de dimensiones descriptivas.

### 2.1 Modelo Estrella

```
         dim_tiempo
              |
dim_colegio — hecho_resultados — dim_estudiante
              |
         dim_geografia
```

- La tabla de hechos está en el centro y contiene las **medidas** (puntuaciones, conteos, montos).
- Las tablas de dimensiones son **planas** (atributos desnormalizados en una sola tabla).
- **Ventajas:** Consultas SQL simples con pocos `JOIN`; rendimiento óptimo en motores columnares.
- **Desventaja:** Redundancia de datos en las dimensiones.

### 2.2 Modelo Copo de Nieve

```
dim_departamento
      |
dim_municipio
      |
dim_colegio — hecho_resultados — dim_estudiante
                                       |
                                 dim_estrato
```

- Las dimensiones están **normalizadas**: se subdividen en sub-dimensiones.
- **Ventajas:** Menos redundancia de almacenamiento; más fácil de actualizar atributos jerárquicos.
- **Desventaja:** Más tablas y `JOIN` complejos que pueden degradar el rendimiento.

### ¿Cuándo usar cuál?

Usa **estrella** cuando la velocidad de consulta es prioritaria y el equipo de BI ejecuta muchas consultas ad-hoc. Usa **copo de nieve** cuando las dimensiones tienen jerarquías naturales profundas (por ejemplo, país → departamento → municipio → barrio) y el espacio de almacenamiento o la consistencia de datos es crítica.

### 🟡 Reflexión 1

Dado el caso de:
- **A)** un sistema de matrículas universitarias en producción, donde estudiantes inscriben materias en tiempo real.
- **B)** un tablero de matrícula histórica del MEN que compara cobertura por departamento en la última década.

¿Cuál es OLTP y cuál OLAP? Justifica en 2-3 frases para cada uno.

_Tu respuesta:_

## 3. Componentes de una Bodega de Datos

1. **Fuentes de datos:** Sistemas OLTP, archivos planos, APIs externas y bases de datos legadas que generan los datos crudos. Son el punto de partida de todo el pipeline.
2. **Staging area:** Zona de aterrizaje temporal donde los datos crudos se copian sin transformar. Permite reintentar cargas sin tocar el origen y sirve como punto de auditoría.
3. **ODS (Operational Data Store) — opcional:** Capa intermedia que integra datos de múltiples fuentes en forma casi-tiempo-real; útil cuando se requiere reporting operacional antes del batch nocturno.
4. **Warehouse (bodega propiamente dicha):** Repositorio central, integrado, orientado al sujeto, histórico y no volátil donde residen los datos limpios y modelados dimensionalmente.
5. **Data marts:** Subconjuntos temáticos del warehouse orientados a un área de negocio (marketing, finanzas, educación). Optimizan el rendimiento para equipos específicos.
6. **Proceso ETL/ELT + orquestador:** Pipeline que **Extrae** datos de las fuentes, los **Transforma** (limpieza, integración, modelado) y los **Carga** en el warehouse. En ELT la transformación ocurre dentro del motor analítico.

El **orquestador** (Apache Airflow, Prefect, Dagster) es el director de orquesta del pipeline: programa las tareas, gestiona dependencias entre ellas, reintenta en caso de fallo, registra el linaje de datos y envía alertas. Sin un orquestador, los pipelines complejos se vuelven frágiles y difíciles de depurar en producción.

## 4. Aplicación a Saber 11: Boceto del Modelo Estrella

A partir del dataset `saber11_muestra_500k.csv` podemos esbozar el siguiente modelo estrella:

### Tabla de hechos: `hecho_resultados`

| Columna | Tipo | Descripción |
|---|---|---|
| `resultado_id` | INT (PK) | Llave subrogada |
| `estudiante_id` | INT (FK) | → dim_estudiante |
| `colegio_id` | INT (FK) | → dim_colegio |
| `tiempo_id` | INT (FK) | → dim_tiempo |
| `geografia_id` | INT (FK) | → dim_geografia |
| `PUNT_LECTURA_CRITICA` | FLOAT | Puntaje Lectura Crítica |
| `PUNT_MATEMATICAS` | FLOAT | Puntaje Matemáticas |
| `PUNT_C_NATURALES` | FLOAT | Puntaje Ciencias Naturales |
| `PUNT_SOCIALES_CIUDADANAS` | FLOAT | Puntaje Sociales y Ciudadanas |
| `PUNT_INGLES` | FLOAT | Puntaje Inglés |
| `PUNT_GLOBAL` | FLOAT | Puntaje Global |

### Dimensiones sugeridas

- **dim_estudiante:** género, estrato, nivel educación padre/madre, acceso a internet, computador en casa.
- **dim_colegio:** nombre, naturaleza (oficial/privado), jornada, calendario, carácter (académico/técnico).
- **dim_tiempo:** periodo (AAAA-S), año, semestre.
- **dim_geografia:** municipio, departamento, zona (urbana/rural).

In [ ]:
import pandas as pd
df = pd.read_csv("../../datos/saber11_muestra_500k.csv")

for col in ["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO",
            "FAMI_ESTRATOVIVIENDA", "FAMI_TIENEINTERNET",
            "COLE_DEPTO_UBICACION", "COLE_MCPIO_UBICACION", "PERIODO"]:
    print(f"{col:35s} → {df[col].nunique():>6} valores únicos")

### 🟡 Reflexión 2

Mirando las cardinalidades del paso anterior:
- ¿Cuáles columnas tienen sentido como dimensión y cuáles no?
- ¿Por qué `COLE_MCPIO_UBICACION` (con más de 1000 valores) puede ser problemática como dimensión plana? ¿Cómo se resolvería usando una jerarquía departamento → municipio?

_Tu respuesta:_

In [ ]:
dim_tiempo = df[["PERIODO"]].drop_duplicates().reset_index(drop=True)
dim_tiempo["tiempo_id"] = dim_tiempo.index + 1
dim_tiempo.head()

### 🟡 Reflexión 3

¿Qué otras dimensiones incluirías si tuvieras acceso a los microdatos crudos completos de Saber 11 (no la muestra)? Piensa en atributos socioeconómicos, escolares y geográficos. ¿Qué información se pierde con el muestreo?

_Tu respuesta:_

## 5. ¿Qué haremos en la Clase 2?

En el **laboratorio de la Clase 2** (miércoles 2-sep / jueves 3-sep) implementarás el modelo dimensional para Saber 11 en tres tareas guiadas:

- **Tarea 1:** Crear las tablas de dimensión (`dim_tiempo`, `dim_colegio`, `dim_geografia`, `dim_estudiante`) a partir del DataFrame de pandas.
- **Tarea 2:** Construir la tabla de hechos (`hecho_resultados`) con llaves foráneas y medidas de puntaje.
- **Tarea 3:** Ejecutar consultas analíticas sobre el modelo dimensional para responder preguntas de negocio (promedio de puntaje global por departamento y periodo, etc.).

En los **últimos 40 minutos** cada grupo de proyecto bosquejará el modelo dimensional para su propio dataset y recibirá retroalimentación del docente antes de comenzar el diseño formal.

### 🟡 Reflexión 4

Bosqueja en 5-6 líneas el modelo dimensional que anticipas para tu dataset del proyecto:
- ¿Cuál será el hecho? ¿Cuál es su granularidad?
- ¿Cuáles serán las 3-4 dimensiones? Nómbralas.
- ¿Alguna dimensión requiere jerarquía (padre-hijo)?

_Tu respuesta:_